In [1]:
import os
import pandas as pd
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")
import re

# base_path = r'C:\Users\IlaBarshilia\OneDrive - OIA Global\VS_Code\FDOT_Invoicing_Automation'
root_dir=r"C:\Users\IlaBarshilia\OneDrive - OIA Global\VS_Code"
base_path=root_dir
folder_paths = [os.path.join(base_path, name) 
                for name in os.listdir(base_path) 
                if os.path.isdir(os.path.join(base_path, name)) and "fdot_downloads" in name]
# Create a DataFrame from folder_paths
df_folders = pd.DataFrame({'folder_path': folder_paths})
# Extract the folder name (after last '\')
df_folders['folder_name'] = df_folders['folder_path'].apply(lambda x: os.path.basename(x))
# Extract the date using regex
df_folders['date_str'] = df_folders['folder_name'].str.extract(r'(\d{4}-[A-Za-z]{3}-\d{2})')
# Convert to datetime for comparison
df_folders['date'] = pd.to_datetime(df_folders['date_str'], format='%Y-%b-%d', errors='coerce')
df_folders
# Find the row with the latest date
# latest_row = df_folders.loc[df_folders['date'].idxmax()]
# latest_folder_path = latest_row['folder_path']
latest_folder_path = df_folders[df_folders['folder_name']=='fdot_downloads_marketshare_2025-Sep-03']['folder_path'].iloc[0]


In [3]:
latest_folder_path

'C:\\Users\\IlaBarshilia\\OneDrive - OIA Global\\VS_Code\\fdot_downloads_marketshare_2025-Sep-03'

In [9]:
all_dataframes = []

for filename in os.listdir(latest_folder_path):
    if filename.endswith('.XLSX') or filename.endswith('.xlsx'):
        file_path = os.path.join(latest_folder_path, filename)
        try:
            df = pd.read_excel(file_path, engine='openpyxl')
            if not df.empty:
                all_dataframes.append(df)
            else:
                print(f"Skipped empty file: {filename}")
        except Exception as e:
            print(f"Error reading {filename}: {e}")

if all_dataframes:
    combined_df = pd.concat(all_dataframes, ignore_index=True)
    combined_df.columns = [col.replace('\n', '').strip() for col in combined_df.columns]
    combined_df=combined_df[['Contract', 'Vendor Name', 'EstimateNumber', 'EstimateEnd Date','Project', 'Unit', 'Item Code', 'Previous', 
                             'This Est.', 'To-Date','Item Description']].copy().drop_duplicates()
    # combined_df['Date_Downloaded'] = latest_row['date_str']
    
    print("All FDOT Downloads of Excel files have been combined into one file.")
else:
    print("No valid Excel files found to combine.")

All FDOT Downloads of Excel files have been combined into one file.


In [10]:
combined_df.shape

(1034496, 11)

In [11]:
df2=combined_df.copy()

In [13]:
df2.shape

(1034496, 11)

In [ ]:
# df1.shape

(822840, 11)

In [14]:
df3=pd.concat([df1,df2])
df3.shape

(1857336, 11)

In [15]:
df3=df3.drop_duplicates()
df3.shape

(1857336, 11)

In [17]:
combined_df.to_csv(r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Source Data\FDOT_Market_Share_Analysis.csv",index=False)

In [16]:
combined_df.to_excel(r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Source Data\FDOT_Market_Share_Analysis.xlsx",index=False)

In [98]:
import pandas as pd
import os
import time
from datetime import datetime

Master_Pay_Item_List=pd.read_excel(root_dir + r"\\FDOT Master Pay Items.xlsx")
Master_Pay_Item_List.columns = Master_Pay_Item_List.iloc[1]
Master_Pay_Item_List = Master_Pay_Item_List.iloc[2:]
Master_Pay_Item_List = Master_Pay_Item_List[Master_Pay_Item_List['Acme Vlookup']== 'Include']
# Create lowercased description columns for merging
# combined_df['Item Description Lower'] = combined_df['Item Description'].str.lower()
# Master_Pay_Item_List['Item Description Lower'] = Master_Pay_Item_List['Item Description'].str.lower()

filtered_df = combined_df.merge(
	Master_Pay_Item_List[['Item Number']],
	left_on=['Item Code'],
	right_on=['Item Number'],
	how='inner'
).drop(columns=['Item Number'])

In [65]:
# Read Cut-off Dates
from datetime import datetime
cutoff_data = pd.read_excel(root_dir + r"\\FDOT Monthly CutOff Dates.xlsx")
current_month = datetime.now().strftime('%B')
last_month = (datetime.now().replace(day=1) - pd.DateOffset(months=1)).strftime('%B')
last_cutoff_date = cutoff_data[cutoff_data['Month'] == last_month]['Cutoff date'].values[0]
current_cutoff_date = cutoff_data[cutoff_data['Month'] == current_month]['Cutoff date'].values[0]
final_df = filtered_df[(filtered_df['EstimateEnd Date']>last_cutoff_date) & (filtered_df['EstimateEnd Date']<=current_cutoff_date)]
yesterday = (datetime.today() - pd.DateOffset(days=1)).strftime('%Y-%b-%d')
# final_df.to_excel(os.path.join(root_dir, 'FDOT_Output_Data_' + str(yesterday) + '.xlsx'), index=False)

In [ ]:
final_df.shape

(1709, 11)

In [70]:
final_df.to_excel(r'C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Source Data\FDOT_Output_Data_2025-Aug-17.xlsx', index=False)

#### Comparison of daily data manual verification

In [61]:
df1 = pd.read_excel(r'C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Source Data\FDOT_Output_Data_2025-Aug-19.xlsx')

df2 = pd.read_excel(r'C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Source Data\FDOT_Output_Data_2025-Aug-20.xlsx')

In [63]:
print("df1 shape", df1.shape)
print("df2 shape", df2.shape)

df1 shape (7670, 11)
df2 shape (8375, 11)


In [68]:
df3=df2.merge(df1, how='outer', on=['Contract', 'Vendor Name', 'EstimateNumber', 'EstimateEnd Date', 'Project', 'Unit', 'Item Code'], indicator=True, suffixes=('_new', '_old'))

df3['_merge'].unique()
# df3.shape

['both', 'left_only', 'right_only']
Categories (3, object): ['left_only', 'right_only', 'both']

In [74]:
# df4=df3[(df3["_merge"]=='left_only') & (~df3['This Est._old'].isna())]
df4=df3[(df3["Contract"]=='T2994') & (df3["Unit"]=='LS')]
# df4=df4[(df4['Contract']=='E1V24') & (df4['EstimateNumber']==12)]
df4

,Contract,Vendor Name,EstimateNumber,EstimateEnd Date,Project,Unit,Item Code,Previous_new,This Est._new,To-Date_new,Item Description_new,Previous_old,This Est._old,To-Date_old,Item Description_old,_merge
4811,T2994,"COXWELL, J.B. CONTRACTING, INC.",3,2025-08-17,446808-2-52-01,LS,0101 1,0.5,0.029,0.529,MOBILIZATION-44680825201,0.5,0.250,0.750,MOBILIZATION-44680825201,both
4812,T2994,"COXWELL, J.B. CONTRACTING, INC.",3,2025-08-17,446808-2-52-01,LS,0102 1,0.0,0.041,0.041,MAINTENANCE OF TRAFFIC-44680825201,0.0,0.041,0.041,MAINTENANCE OF TRAFFIC-44680825201,both
4813,T2994,"COXWELL, J.B. CONTRACTING, INC.",3,2025-08-17,446808-2-52-01,LS,0102 4 1,0.0,0.800,0.800,PEDESTRIAN OR BICYCLE SPECIAL DETOUR-44680825201,0.0,0.800,0.800,PEDESTRIAN OR BICYCLE SPECIAL DETOUR-44680825201,both
4814,T2994,"COXWELL, J.B. CONTRACTING, INC.",3,2025-08-17,446808-2-52-02,LS,0101 1,0.5,0.028,0.528,MOBILIZATION-44680825202,0.5,0.250,0.750,MOBILIZATION-44680825202,both


In [13]:
df2.shape

(3726, 11)

In [14]:
df1.shape

(1708, 11)

#### Comparison of daily data through code

In [21]:
import pandas as pd

# Load the Excel file
file_path = r'C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Output Files\September_2025_Cutoff\FDOT_Output_Data_2025-Sep-21.xlsx'
df = pd.read_excel(file_path)
print(df.columns)
df1 = df.drop(columns=['EstimateEnd Date', 'Item Description'])
# Find duplicate rows (entire row match)
duplicates = df1[df1.duplicated()]

# Print or save the duplicates
print("Duplicate rows:")
duplicates

Index(['Contract', 'EstimateNumber', 'EstimateEnd Date', 'Project', 'Unit',
       'Item Code', 'Previous', 'This Est.', 'To-Date', 'Item Description'],
      dtype='object')
Duplicate rows:


,Contract,EstimateNumber,Project,Unit,Item Code,Previous,This Est.,To-Date
27,E3X03,1,406196-3-52-01,LS,0101 1,1.000,0.0,1.000
29,E3X03,1,406196-3-52-01,LS,0102 1,1.000,0.0,1.000
31,E3X03,1,406196-3-52-01,GM,0102913 22,0.058,0.0,0.058
33,E3X03,1,406196-3-52-01,EA,0706 1 3,8.000,0.0,8.000
166,E4X26,5,447667-1-52-01,LS,0101 1,1.000,0.0,1.000
...,...,...,...,...,...,...,...,...
1426,T6586,7,443908-1-52-01,EA,0536 85 20,2.000,0.0,2.000
1428,T6586,7,443908-1-52-01,EA,0536 85 24,4.000,0.0,4.000
1584,T7472,33,431821-2-52-01,LS,0101 1,1.000,0.0,1.000
1587,T7472,33,443770-1-52-01,LS,0101 1,1.000,0.0,1.000


In [ ]:
import os
import pandas as pd
import re
from datetime import datetime

# 🔧 Replace with your actual folder path
folder_path = r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Output Files\September_2025_Cutoff"
output_file = r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Output Files\FDOT_Daily_Comparison_Results_Sep19_CutOff.xlsx"

# 📅 Regex to extract date in format '2025-Aug-17'
date_pattern = re.compile(r'(\d{4}-[A-Za-z]{3}-\d{2})')

# 📂 Step 1: Collect and sort files by date
files = []
for filename in os.listdir(folder_path):
    match = date_pattern.search(filename)
    if match:
        try:
            file_date = datetime.strptime(match.group(1), "%Y-%b-%d")
            files.append((file_date, filename))
        except ValueError:
            print(f"⚠️ Skipping file with invalid date format: {filename}")

files.sort()  # Sort by date

# 🔑 Key columns for identifying rows
key_columns = ["Contract", "EstimateNumber", "Project", "Unit", "Item Code", "Previous"]
all_summaries = []

# 📊 Step 2: Compare each file with the next one and export results
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    for i in range(len(files) - 1):
        date1, file1 = files[i]
        date2, file2 = files[i + 1]

        df1 = pd.read_excel(os.path.join(folder_path, file1), engine='openpyxl')
        print(df1.columns)
        df1 = df1.drop(columns=['EstimateEnd Date', 'Item Description'])
        df1=df1.drop_duplicates()
        print(df1.shape)
        df2 = pd.read_excel(os.path.join(folder_path, file2), engine='openpyxl')
        df2 = df2.drop(columns=['EstimateEnd Date', 'Item Description'])
        df2=df2.drop_duplicates()
        print(df2.shape)


        # Convert all values to string for consistent comparison
        df1_str = df1.astype(str).drop_duplicates()
        df2_str = df2.astype(str).drop_duplicates()

        # ✅ New rows: full rows in df2 not in df1
        new_mask = ~df2_str.apply(tuple, axis=1).isin(df1_str.apply(tuple, axis=1))
        new_rows = df2_str[new_mask]

        # 🗑️ Removed rows: full rows in df1 not in df2
        removed_mask = ~df1_str.apply(tuple, axis=1).isin(df2_str.apply(tuple, axis=1))
        old_rows = df1_str[removed_mask]

        # 🔄 Changed rows: same keys, different content
        df1_keyed = df1_str.set_index(key_columns)
        df2_keyed = df2_str.set_index(key_columns)

        common_index = df1_keyed.index.intersection(df2_keyed.index)
        df1_common = df1_keyed.loc[common_index]
        df2_common = df2_keyed.loc[common_index]

        # Align columns before comparison
        df2_common_aligned = df2_common.reindex(columns=df1_common.columns)

        # Compare aligned DataFrames
        changed_mask = (df1_common != df2_common_aligned).any(axis=1)
        changed_rows = df2_common_aligned[changed_mask].reset_index()

        # Remove changed rows from new and old tabs
        changed_tuples = changed_rows.apply(tuple, axis=1)

        # Convert new and old rows to tuples for comparison
        new_tuples = new_rows.apply(tuple, axis=1)
        old_tuples = old_rows.apply(tuple, axis=1)

        # Filter out rows that are also in changed_rows
        new_rows_filtered = new_rows[~new_tuples.isin(changed_tuples)]
        old_rows_filtered = old_rows[~old_tuples.isin(changed_tuples)]

        
        # 📝 Write to Excel with sheet names based on newer file's date
        sheet_date = date2.strftime('%Y-%b-%d')
        new_rows_filtered.to_excel(writer, sheet_name=f"{sheet_date}_new"[:31], index=False)
        old_rows_filtered.to_excel(writer, sheet_name=f"{sheet_date}_removed"[:31], index=False)
        changed_rows.to_excel(writer, sheet_name=f"{sheet_date}_changed"[:31], index=False)
        all_summaries.append({
            "Compared Files": [f"{file1} → {file2}"],
            "Date": [sheet_date],
            "New Rows": [len(new_rows_filtered)],
            "Removed Rows": [len(old_rows_filtered)],
            "Changed Rows": [len(changed_rows)]
        })

        print(f"✅ Compared {file1} → {file2}")
        print(f"   ➕ New: {len(new_rows)} | 🗑️ Removed: {len(old_rows)} | 🔄 Changed: {len(changed_rows)}")
        summary_df = pd.DataFrame(all_summaries)
        summary_df.to_excel(writer, sheet_name="Summary", index=False)

print(f"\n✅ All comparisons complete. Results saved to: {output_file}")

Index(['Contract', 'EstimateNumber', 'EstimateEnd Date', 'Project', 'Unit',
       'Item Code', 'Previous', 'This Est.', 'To-Date', 'Item Description'],
      dtype='object')
(1630, 8)
(3925, 8)
✅ Compared FDOT_Output_Data_2025-Sep-21.xlsx → FDOT_Output_Data_2025-Sep-22.xlsx
   ➕ New: 2320 | 🗑️ Removed: 25 | 🔄 Changed: 4
Index(['Contract', 'EstimateNumber', 'EstimateEnd Date', 'Project', 'Unit',
       'Item Code', 'Previous', 'This Est.', 'To-Date', 'Item Description'],
      dtype='object')
(3925, 8)
(7125, 8)
✅ Compared FDOT_Output_Data_2025-Sep-22.xlsx → FDOT_Output_Data_2025-Sep-23.xlsx
   ➕ New: 3275 | 🗑️ Removed: 75 | 🔄 Changed: 12


Exception ignored in: <function ZipFile.__del__ at 0x0000020879549940>
Traceback (most recent call last):
  File "c:\Users\IlaBarshilia\AppData\Local\Programs\Python\Python313\Lib\zipfile\__init__.py", line 1988, in __del__
    self.close()
  File "c:\Users\IlaBarshilia\AppData\Local\Programs\Python\Python313\Lib\zipfile\__init__.py", line 2005, in close
    self.fp.seek(self.start_dir)
ValueError: seek of closed file


Index(['Contract', 'EstimateNumber', 'EstimateEnd Date', 'Project', 'Unit',
       'Item Code', 'Previous', 'This Est.', 'To-Date', 'Item Description'],
      dtype='object')
(7125, 8)
(8217, 8)
✅ Compared FDOT_Output_Data_2025-Sep-23.xlsx → FDOT_Output_Data_2025-Sep-24.xlsx
   ➕ New: 1187 | 🗑️ Removed: 95 | 🔄 Changed: 1
Index(['Contract', 'EstimateNumber', 'EstimateEnd Date', 'Project', 'Unit',
       'Item Code', 'Previous', 'This Est.', 'To-Date', 'Item Description'],
      dtype='object')
(8217, 8)
(8437, 8)
✅ Compared FDOT_Output_Data_2025-Sep-24.xlsx → FDOT_Output_Data_2025-Sep-25.xlsx
   ➕ New: 225 | 🗑️ Removed: 5 | 🔄 Changed: 1

✅ All comparisons complete. Results saved to: C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Output Files\September_2025_Cutoff\FDOT_Daily_Comparison_Results_Sep19_CutOff.xlsx


#### Compare Estimate Date for Each Contract and latest Estimate Number

In [92]:
latest_df = pd.read_excel(r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Output Files\FDOT_Output_Data_2025-Aug-26.xlsx")
grouped_df=latest_df.groupby("Contract")["EstimateNumber"].max().reset_index()

grouped_df = grouped_df.merge(latest_df[['Contract', 'EstimateNumber', 'EstimateEnd Date']], how='left', on=['Contract', 'EstimateNumber']).drop_duplicates()
grouped = grouped_df.groupby(['Contract', 'EstimateNumber'])['EstimateEnd Date']
grouped_df['Count_EndDates'] = grouped.transform('nunique')
grouped_df['EstimateEnd Date'] = grouped.transform(lambda x:list(set(x)))
grouped_df=grouped_df.reset_index(drop=True)
trial=grouped_df[grouped_df['Count_EndDates']>1]
trial

,Contract,EstimateNumber,EstimateEnd Date,Count_EndDates
96,T2844,31,2025-07-16,3
97,T2844,31,2025-07-18,3
98,T2844,31,2025-07-17,3
135,T2A02,7,2025-07-30,2
136,T2A02,7,2025-07-31,2


##### Comparison without cut off date filter

In [113]:
grouped_df=filtered_df.groupby("Contract")["EstimateNumber"].max().reset_index()

grouped_df = grouped_df.merge(filtered_df[['Contract', 'EstimateNumber', 'EstimateEnd Date']], how='left', on=['Contract', 'EstimateNumber']).drop_duplicates()
grouped = grouped_df.groupby(['Contract', 'EstimateNumber'])['EstimateEnd Date']
grouped_df['Count_EndDates'] = grouped.transform('nunique')
grouped_df['EstimateEnd Date'] = grouped.transform(lambda x:list(set(x)))
grouped_df=grouped_df.reset_index(drop=True)
grouped_df = grouped_df.groupby(['Contract', 'EstimateNumber']).agg(max_date_diff_days = ('EstimateEnd Date', lambda x: (x.max() - x.min()).days)).reset_index()
grouped_df = grouped_df[grouped_df['max_date_diff_days']>=30]
grouped_df

,Contract,EstimateNumber,max_date_diff_days
24,E3V77,15,54
34,E3W54,8,65
53,E4U88,23,51
79,E53F7,4,35
101,E7N96,16,198
119,E8U91,12,30
171,T2907,23,159
231,T3711,35,35
237,T3735,42,149
243,T3783,10,35


In [114]:
# grouped_df.to_excel('Check_EstimateEnd Dates.xlsx', index=False)

filtered_df=filtered_df.merge(grouped_df, how='inner', on=['Contract', 'EstimateNumber'])
filtered_df.to_excel('Check_Estimate End Dates more than 30 days for latest estimate number.xlsx', index=False)

In [ ]:

trial =grouped_df[grouped_df['EstimateEnd Date']>=30]
trial

,Contract,EstimateNumber,EstimateEnd Date
24,E3V77,15,54
34,E3W54,8,65
53,E4U88,23,51
79,E53F7,4,35
101,E7N96,16,198
119,E8U91,12,30
171,T2907,23,159
231,T3711,35,35
237,T3735,42,149
243,T3783,10,35


In [ ]:
trial['Contract'].nunique()
# filtered_df.to_excel('Check_output.xlsx', index=False)
trial.to_excel

111